In [ ]:
import os
import random
import torch
import json
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch import amp
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
from contextlib import nullcontext

if torch.cuda.is_available(): DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available(): DEVICE = torch.device("mps")
else: DEVICE = torch.device("cpu")

SEED        = 1337

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CACHE_ROOT  = "../../data/dataset/classifier_mel"
CHECKPOINTS = "../../data/checkpoints"

with open(f"{CACHE_ROOT}/index.json") as f:
    IDX = json.load(f)

CLASSES = tuple(IDX["classes"])
ITEMS   = IDX["items"]
P       = IDX.get("params", {})
DB_LO   = float(P.get("top_db", 80)) * -1 if "top_db" in P else -90.0
DB_HI   = 0.0
FPS     = P.get("sr", 44100) / P.get("hop", 16)
T_SEC   = 6.0
CROP_F  = int(FPS * T_SEC)
TIME_D  = 4
FREQ_D  = 2
BATC_S  = 8
EPOCHS  = 5
LR      = 3e-4
WD      = 1e-4
PATIEN  = 6
LAB_SM  = 0.1
ACC_ST  = 2

print(f"Using {DEVICE} | Number of classes: {len(CLASSES)} | Number of items: {len(ITEMS)}")

In [ ]:
def _norm_db(x):
    return (x - DB_LO) / (DB_HI - DB_LO + 1e-8)

In [ ]:
class MelDataset(Dataset):
    def __init__(self, items, train=True):
        self.items = items
        self.train = train
    
    def __len__(self):
        return len(self.items)
    

    def _rand_crop_cols(self, T):
        if T <= CROP_F:
            return 0, T
        s = random.randint(0, T - CROP_F)
        return s, s+ CROP_F
        
    def __getitem__(self, i):
        it = self.items[i]
        X = np.load(it["mel_path"]).astype(np.float32)
        X = _norm_db(X)
        M, T = X.shape

        if self.train:
            s, e = self._rand_crop_cols(T)
        else:
            s, e = (max(0, T//2 - CROP_F//2), min(T, T//2 + CROP_F//2))
        
        X = X[:, s:e]

        if X.shape[1] < CROP_F:
            pad = CROP_F - X.shape[1]
            X = np.pad(X, ((0,0), (0,pad)), mode='edge')

        X = torch.from_numpy(X).unsqueeze(0)

        if self.train:
            t_w = int(0.08 * X.shape[-1])

            if t_w > 0:
                t0 = random.randint(0, max(0, X.shape[-1] - t_w))
                X[:,:,t0:t0+t_w] *= 0.0

            f_w = int(0.06 * X.shape[-2])
            
            if f_w > 0:
                f0 = random.randint(0, max(0, X.shape[-2]-f_w))

        if FREQ_D > 1 or TIME_D > 1:
            X = F.avg_pool2d(X, kernel_size=(FREQ_D, TIME_D), stride=(FREQ_D, TIME_D))

        y = it["class_idx"]
        
        return X, y
    
labels = [it["class_idx"] for it in ITEMS]
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
(tr_idx, va_idx), = sss.split(np.zeros(len(labels)), labels)
train_items = [ITEMS[i] for i in tr_idx]
val_items   = [ITEMS[i] for i in va_idx]


tr_ds = MelDataset(train_items, train=True)
va_ds = MelDataset(val_items,   train=False)
trL = DataLoader(tr_ds, batch_size=BATC_S, shuffle=True,  num_workers=0, pin_memory=False)
vaL = DataLoader(va_ds, batch_size=BATC_S, shuffle=False, num_workers=0, pin_memory=False)


sample_in, _ = tr_ds[0]
print("Input patch shape to CNN:", tuple(sample_in.shape))

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c, c//r, 1), nn.SiLU(),
            nn.Conv2d(c//r, c, 1), nn.Sigmoid()
        )

    def forward(self, x):
        w = self.fc(x)
        return x * w
    
def conv_bn(in_c, out_c, k=3, s=1, p=1):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, k, s, p, bias=False),
        nn.BatchNorm2d(out_c),
        nn.SiLU(inplace=True)
    )

class ResBlock(nn.Module):
    def __init__(self, c, down=False):
        super().__init__()
        s = 2 if down else 1
        self.conv1 = conv_bn(c, c, 3, s, 1)
        self.conv2 = nn.Sequential(
            nn.Conv2d(c, c, 3, 1, 1, bias=False),
            nn.BatchNorm2d(c)
        )
        self.se = SEBlock(c)
        self.down = nn.AvgPool2d(2) if down else nn.Identity()
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        idt = self.down(x) if isinstance(self.down, nn.AvgPool2d) else x
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.se(out)
        if out.shape == idt.shape:
            out = out + idt
        return self.act(out)


class ResMelNet(nn.Module):
    def __init__(self, n_classes: int):
        super().__init__()
        self.stem = nn.Sequential(
            conv_bn(1, 64, 5, 2, 2), nn.MaxPool2d(2)
        )
        self.stage1 = nn.Sequential(*[ResBlock(64) for _ in range(3)])
        self.stage2 = nn.Sequential(ResBlock(64, down=True), *[ResBlock(64) for _ in range(3)])
        self.stage3 = nn.Sequential(ResBlock(128, down=True), *[ResBlock(128) for _ in range(3)])
        self.stage4 = nn.Sequential(ResBlock(256, down=True), *[ResBlock(256) for _ in range(3)])
        self.proj2 = nn.Sequential(nn.Conv2d(64, 128, kernel_size=1, bias=False), nn.BatchNorm2d(128), nn.SiLU(inplace=True))
        self.proj3 = nn.Sequential(nn.Conv2d(128, 256, kernel_size=1, bias=False), nn.BatchNorm2d(256), nn.SiLU(inplace=True))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, 256), nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        
        x = self.stage2(x)
        x = self.proj2(x)

        x = self.stage3(x)
        x = self.proj3(x)

        x = self.stage4(x)
        return self.head(x)
    
model = ResMelNet(n_classes=len(CLASSES)).to(DEVICE)


In [ ]:
class LSCELoss(nn.Module):
    def __init__(self, eps=0.1):
        super().__init__()
        self.eps = eps
    def forward(self, logits, y):
        num_classes = logits.size(-1)
        y_onehot = torch.zeros_like(logits).scatter_(1, y.unsqueeze(1), 1)
        y_smooth = (1 - self.eps) * y_onehot + self.eps / num_classes
        logp = F.log_softmax(logits, dim=1)
        return -(y_smooth * logp).sum(dim=1).mean()


opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=4, T_mult=2)
lossf  = LSCELoss(LAB_SM)
scaler = amp.GradScaler(enabled=(DEVICE.type == "cuda"))

best_acc = 0.0
noimp = 0
hist = {"tr_loss": [], "va_acc": []}


for ep in range(EPOCHS):
    model.train()
    run, steps = 0.0, 0
    opt.zero_grad(set_to_none=True)

    autocast_ctx = amp.autocast(device_type="cuda", dtype=torch.float16) if DEVICE.type == "cuda" else nullcontext()

    train_bar = tqdm(total=len(trL), desc=f"Train epoch {ep+1}/{EPOCHS}", mininterval=0.5, leave=False)
    for step, (X, y) in enumerate(trL, 1):
        X = X.to(DEVICE, non_blocking=True)
        y = torch.as_tensor(y, device=DEVICE)
        with autocast_ctx:
            logits = model(X)
            loss = lossf(logits, y) / ACC_ST

        if DEVICE.type == "cuda":
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if step % ACC_ST == 0:
            if DEVICE.type == "cuda":
                scaler.step(opt)
                scaler.update()
            else:
                opt.step()
            opt.zero_grad(set_to_none=True)

        run += loss.item() * ACC_ST
        steps += 1
        train_bar.update(1)
    train_bar.close()

    tr_loss = run / max(1, steps)
    sched.step(ep + 1)
    hist["tr_loss"].append(tr_loss)

    model.eval()
    correct = n = 0
    y_true, y_pred = [], []

    val_bar = tqdm(total=len(vaL), desc=f"Valid epoch {ep+1}/{EPOCHS}", mininterval=0.5, leave=False)
    with torch.no_grad():
        for X, y in vaL:
            X = X.to(DEVICE, non_blocking=True)
            y = torch.as_tensor(y, device=DEVICE)
            p = model(X).argmax(1)
            correct += (p == y).sum().item()
            n += y.numel()
            y_true += y.tolist()
            y_pred += p.tolist()
            val_bar.update(1)
    val_bar.close()

    acc = correct / max(1, n)
    hist["va_acc"].append(acc)
    print(f"[Epoch {ep+1:02d}] train_loss={tr_loss:.4f} | val_acc={acc:.3f}")

    if acc > best_acc:
        best_acc, noimp = acc, 0
        os.makedirs(CHECKPOINTS, exist_ok=True)
        torch.save({"model": model.state_dict(), "classes": CLASSES}, f"{CHECKPOINTS}/cnn.genre.pth")
        print(f"Saved new best model — val_acc={best_acc:.3f}")
    else:
        noimp += 1
        if noimp >= PATIEN:
            print("Early stopping.")
            break

print("Best val_acc:", round(best_acc, 3))

In [ ]:
fig, ax1 = plt.subplots(figsize=(7,4))
ax1.plot(hist["tr_loss"], label="train loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss")
ax2 = ax1.twinx()
ax2.plot(hist["va_acc"], "g--", label="val acc")
ax2.set_ylabel("accuracy")
ax1.grid(True, alpha=0.25)
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines+lines2, labels+labels2, loc="best")
plt.title("Training curves")
plt.show()